# 06 — Independent PPO agent: Tasks 1–4

This notebook records the implemented and executed PPO curriculum. It coexists with `06_linear_q_final_stopline.ipynb`; no previous notebook was overwritten. PPO uses the current 22-feature DDN action-relative representation (132 flattened inputs), the unchanged safe-action mask and DDQN reward shaping.

## Implementation and checks

The model has a shared 128×128 MLP, a six-logit actor and scalar critic. Training uses masked categorical sampling, fresh on-policy rollouts, GAE(γ=.99, λ=.95), clipped PPO (ε=.2), four epochs, 64-sample minibatches, entropy coefficient .01 and gradient clipping .5. Evaluation uses deterministic masked argmax. There is no replay buffer or target network.

Unit checks cover zero illegal-action probability, probability normalization, fresh-policy ratio=1, terminal GAE, hand-calculated two-step GAE, and positive/negative clipping. A real three-round training smoke test also verified framework callback order and terminal handling.

## Executed curriculum

| stage | training | held-out evaluation | checkpoint |
|---|---|---|---|
| Task 1 | 200 coin-heaven rounds | 20 rounds, seed 61291 | `frozen_ppo_task1` |
| Task 2 | 100 classic solo rounds | 20 rounds, seed 62091 | `frozen_ppo_task2` |
| Task 3 | 80 classic rounds vs 3 peaceful agents | 20 rounds, seed 63091 | `frozen_ppo_task3` |
| Task 4 | 50 rule + 50 mixed rounds | 5×30 rule and 5×30 mixed, seeds 32001–32005 | `frozen_ppo_final` |

Total curriculum training: 480 rounds plus a three-round smoke test. Total reported evaluation: 360 rounds.

## Held-out results

| task/lineup | rounds | score | coins | crates | kills | suicides | survival | win |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Task 1 coin-heaven | 20 | 23.50 | 23.50 | 0 | 0 | 0 | 1.00 | — |
| Task 2 classic solo | 20 | 0.90 | 0.90 | 22.50 | 0 | 0 | 1.00 | — |
| Task 3 peaceful | 20 | 4.30 | 2.05 | 29.85 | 0.45 | 0.15 | 0.85 | 1.00 |
| Task 4 rule-based | 150 | 2.333 | 0.867 | 17.307 | 0.293 | 0.647 | 0.307 | 0.18 |
| Task 4 mixed | 150 | 2.587 | 0.953 | 20.993 | 0.327 | 0.427 | 0.527 | 0.14 |

In [1]:
from pathlib import Path
import sys
def find_project_root(start=None):
    candidate = Path(start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / 'main.py').is_file() and (directory / 'agent_code').is_dir():
            return directory
    raise FileNotFoundError('Could not find the project root containing main.py and agent_code/')

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
from evaluation.summarize_stats import summarize_match
task_paths = {1: ROOT / 'results/ppo_task1_eval_after200.json', 2: ROOT / 'results/ppo_task2_eval.json', 3: ROOT / 'results/ppo_task3_eval.json'}
for task, path in task_paths.items():
    print(task, next(r for r in summarize_match([path]) if r['agent'] == 'ppo_agent'))
for lineup in ('rule_based', 'mixed'):
    paths = [ROOT / f'results/ppo_final_{lineup}_seed{s}.json' for s in range(32001, 32006)]
    print(lineup, next(r for r in summarize_match(paths) if r['agent'] == 'frozen_ppo_final'))

1 {'agent': 'ppo_agent', 'rounds': 20, 'mean_score': 23.5, 'win_rate': nan, 'mean_rank': nan, 'mean_score_margin': nan, 'coins_per_round': 23.5, 'crates_per_round': 0.0, 'bombs_per_round': 58.0, 'kills_per_round': 0.0, 'suicides_per_round': 0.0, 'invalid_per_round': 0.0, 'survival_rate': 1.0, 'mean_action_ms': 1.3563624322414398}
2 {'agent': 'ppo_agent', 'rounds': 20, 'mean_score': 0.9, 'win_rate': nan, 'mean_rank': nan, 'mean_score_margin': nan, 'coins_per_round': 0.9, 'crates_per_round': 22.5, 'bombs_per_round': 44.35, 'kills_per_round': 0.0, 'suicides_per_round': 0.0, 'invalid_per_round': 0.0, 'survival_rate': 1.0, 'mean_action_ms': 0.9267972111701965}
3 {'agent': 'ppo_agent', 'rounds': 20, 'mean_score': 4.3, 'win_rate': 1.0, 'mean_rank': 1.0, 'mean_score_margin': 4.266666666666667, 'coins_per_round': 2.05, 'crates_per_round': 29.85, 'bombs_per_round': 35.8, 'kills_per_round': 0.45, 'suicides_per_round': 0.15, 'invalid_per_round': 0.7, 'survival_rate': 0.85, 'mean_action_ms': 1.1764

## Diagnostics and interpretation

The frozen final checkpoint contains 181,005 environment steps and 8,668 optimizer minibatch updates. All parameters are finite. Its last stored update reports policy loss .126, value loss 793.19, entropy .384, approximate KL .0139 and clip fraction .022. The KL and clip fraction are controlled, while the large critic loss signals difficult/non-stationary return fitting.

PPO learns progressively (Task-1 coins rose from 6.6 after 50 rounds to 23.5 after roughly 200), completes Task 2 safely, and beats passive Task-3 opponents. It does not pass the intended full Task-1 capability gate and performs poorly in Task 4: high suicide rates and scores well below the frozen DDQN/linear baselines. This is a valid working PPO baseline, not a converged competition model. More on-policy samples, vectorized environments, reward normalization and checkpoint selection would be required before a definitive architectural comparison.

## Formal continuation from frozen checkpoints

The baseline was continued from `frozen_ppo_task1`, with training and evaluation seeds kept disjoint. Capability gates select checkpoints; evaluation games are never replayed into PPO. This continuation adds **471 training rounds** and **390 held-out validation rounds**. Every candidate is frozen independently.

| candidate | continuation data | decision |
|---|---|---|
| Stage 1 | +200 coin-heaven | retain, but Task 1 plateaued |
| Stage 2A contiguous | +80 coin then +20 loot-crate | reject: ordering caused catastrophic forgetting |
| Stage 2A interleaved | 5 × (8 coin + 2 crate) | retain: simultaneous improvement |
| Stage 2B | 5 × (4 coin + 6 crate) | **promote**: passed the ~40/50 Task-1 gate |
| Stage 3 | 5 × (2 coin + 3 crate + 5 peaceful) | reject: early-task forgetting |
| Stage 4 branch | 3 short retention/rule/mixed cycles from Stage 2B | reject: Task 1 and Task 4 regressed |

The important correction was to use short alternating blocks. A contiguous 80/20 sequence is only a ratio in aggregate; its last environment can dominate the latest on-policy updates.

In [2]:
continued = [
    ('stage1/T1', 'results/ppo07_stage1_task1_val.json', 'frozen_ppo_curriculum_stage1'),
    ('stage2A-contiguous/T1', 'results/ppo07_stage2a_task1_val.json', 'frozen_ppo_curriculum_stage2a'),
    ('stage2A-contiguous/T2', 'results/ppo07_stage2a_task2_val.json', 'frozen_ppo_curriculum_stage2a'),
    ('stage2A-interleaved/T1', 'results/ppo07_stage2ai_task1_val.json', 'frozen_ppo_curriculum_stage2a_interleaved'),
    ('stage2A-interleaved/T2', 'results/ppo07_stage2ai_task2_val.json', 'frozen_ppo_curriculum_stage2a_interleaved'),
    ('stage2B/T1', 'results/ppo07_stage2b_task1_val.json', 'frozen_ppo_curriculum_stage2b'),
    ('stage2B/T2', 'results/ppo07_stage2b_task2_val.json', 'frozen_ppo_curriculum_stage2b'),
    ('stage3/T1', 'results/ppo07_stage3_task1_val.json', 'frozen_ppo_curriculum_stage3'),
    ('stage3/T2', 'results/ppo07_stage3_task2_val.json', 'frozen_ppo_curriculum_stage3'),
    ('stage3/T3', 'results/ppo07_stage3_task3_val.json', 'frozen_ppo_curriculum_stage3'),
    ('stage4/T1', 'results/ppo07_stage4_task1_val.json', 'frozen_ppo_curriculum_stage4'),
    ('stage4/T2', 'results/ppo07_stage4_task2_val.json', 'frozen_ppo_curriculum_stage4'),
    ('stage4/T3', 'results/ppo07_stage4_task3_val.json', 'frozen_ppo_curriculum_stage4'),
    ('stage4/rule', 'results/ppo07_stage4_rule_val.json', 'frozen_ppo_curriculum_stage4'),
    ('stage4/mixed', 'results/ppo07_stage4_mixed_val.json', 'frozen_ppo_curriculum_stage4'),
]
rows = []
for label, relpath, agent in continued:
    result = next(r for r in summarize_match([ROOT / relpath]) if r['agent'] == agent)
    rows.append({k: result[k] for k in ('rounds', 'mean_score', 'coins_per_round', 'crates_per_round', 'kills_per_round', 'suicides_per_round', 'invalid_per_round', 'survival_rate')})
    rows[-1]['candidate'] = label
columns = ('candidate', 'rounds', 'mean_score', 'coins_per_round', 'crates_per_round', 'kills_per_round', 'suicides_per_round', 'invalid_per_round', 'survival_rate')
print(' | '.join(columns))
for row in rows:
    print(' | '.join(str(round(row[c], 3)) if isinstance(row[c], float) else str(row[c]) for c in columns))

candidate | rounds | mean_score | coins_per_round | crates_per_round | kills_per_round | suicides_per_round | invalid_per_round | survival_rate
stage1/T1 | 30 | 23.467 | 23.467 | 0.0 | 0.0 | 0.0 | 0.0 | 1.0
stage2A-contiguous/T1 | 30 | 11.233 | 11.233 | 0.0 | 0.0 | 0.0 | 0.0 | 1.0
stage2A-contiguous/T2 | 30 | 8.767 | 8.767 | 28.867 | 0.0 | 0.0 | 0.0 | 1.0
stage2A-interleaved/T1 | 30 | 24.467 | 24.467 | 0.0 | 0.0 | 0.0 | 0.0 | 1.0
stage2A-interleaved/T2 | 30 | 9.8 | 9.8 | 31.0 | 0.0 | 0.0 | 0.0 | 1.0
stage2B/T1 | 30 | 42.067 | 42.067 | 0.0 | 0.0 | 0.0 | 0.0 | 1.0
stage2B/T2 | 30 | 8.333 | 8.333 | 27.167 | 0.0 | 0.0 | 0.0 | 1.0
stage3/T1 | 20 | 11.35 | 11.35 | 0.0 | 0.0 | 0.0 | 0.0 | 1.0
stage3/T2 | 20 | 6.7 | 6.7 | 27.85 | 0.0 | 0.0 | 0.0 | 1.0
stage3/T3 | 20 | 2.05 | 0.55 | 15.35 | 0.3 | 0.0 | 0.45 | 1.0
stage4/T1 | 20 | 12.65 | 12.65 | 0.0 | 0.0 | 0.0 | 0.0 | 1.0
stage4/T2 | 20 | 7.2 | 7.2 | 27.4 | 0.0 | 0.0 | 0.0 | 1.0
stage4/T3 | 20 | 4.25 | 1.5 | 27.85 | 0.55 | 0.0 | 1.75 | 1.0
sta

## Continuation result and checkpoint selection

| checkpoint / task | score | coins | crates | kills | suicides | invalid |
|---|---:|---:|---:|---:|---:|---:|
| Stage 1 / T1 | 23.47 | 23.47 | 0 | 0 | 0 | 0 |
| contiguous Stage 2A / T1 | 11.23 | 11.23 | 0 | 0 | 0 | 0 |
| contiguous Stage 2A / T2 | 8.77 | 8.77 | 28.87 | 0 | 0 | 0 |
| interleaved Stage 2A / T1 | 24.47 | 24.47 | 0 | 0 | 0 | 0 |
| interleaved Stage 2A / T2 | 9.80 | 9.80 | 31.00 | 0 | 0 | 0 |
| **Stage 2B / T1** | **42.07** | **42.07** | 0 | 0 | 0 | 0 |
| **Stage 2B / T2** | **8.33** | **8.33** | **27.17** | 0 | 0 | 0 |
| Stage 3 / T3 | 2.05 | 0.55 | 15.35 | 0.30 | 0 | 0.45 |
| Stage 4 / T3 | 4.25 | 1.50 | 27.85 | 0.55 | 0 | 1.75 |
| Stage 4 / rule | 1.73 | 0.73 | 16.53 | 0.20 | 0.63 | 0.73 |
| Stage 4 / mixed | 1.17 | 0.50 | 20.17 | 0.13 | 0.37 | 0.63 |

`frozen_ppo_curriculum_best` is a byte-identical copy of Stage 2B. It is the best new general capability checkpoint: Task 1 improved from 23.5 to 42.07 while Task 2 remained suicide-free. Stage 3 and Stage 4 are diagnostic candidates, not promoted models. For Task 4 specifically, the older `frozen_ppo_final` remains better (2.333 rule / 2.587 mixed) than the new Stage-4 branch.

The experiment therefore finds **real remaining optimization headroom in Tasks 1–2**, but also a persistent multi-task stability limit. More rounds alone are not justified for Tasks 3–4. The next useful change should target training mechanics—randomized/vectorized environment mixing, advantage/reward normalization, smaller learning rate during opponent stages, and best-checkpoint selection at shorter intervals—before another large run.

In [3]:
import hashlib
best = ROOT / 'agent_code/frozen_ppo_curriculum_best/ppo_model.pt'
source = ROOT / 'agent_code/frozen_ppo_curriculum_stage2b/ppo_model.pt'
digest = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert best.exists() and digest(best) == digest(source)
{'selected_checkpoint': str(best.relative_to(ROOT)), 'sha256': digest(best), 'selection': 'Stage 2B held-out capability gates'}

{'selected_checkpoint': 'agent_code/frozen_ppo_curriculum_best/ppo_model.pt',
 'sha256': '39021a1a26785443f7777d7d068c54d6f1baa1336ecfea8ecdd56cb761cc3a52',
 'selection': 'Stage 2B held-out capability gates'}